# Lab 5: Generative Modeling — VAE and Diffusion

In this lab you will train and compare two compact generative models on
FashionMNIST:

1. a variational autoencoder (VAE), where you will tune latent dimension and
   the KL weight \(\beta\); and
2. a diffusion denoiser, where you will compare valid DDIM sampling budgets.

The implementations are complete. Your job is to change exposed settings,
run controlled comparisons, inspect the resulting images, and explain the
tradeoffs. This is the experimental bridge from Module 5's ELBO and diffusion
equations to the Final Project.

**Grading.** The only numeric value submitted to edX is a deterministic KL
diagnostic computed from fixed printed inputs. Training losses, runtimes, and
sample-quality judgments are useful evidence, but are not auto-graded because
they can vary across accelerators and software versions.

Adapted from *6.7960 Fall 2024 HW5 generative.ipynb*, Questions 4–6 and 11–12.
The beta sweep and reduced-step DDIM comparison are tuning-oriented online
extensions.

## 1. Setup and run mode

Start with **MODE = "quick"**. It uses a deterministic subset and compact
training budgets so that the whole notebook is practical on a free Colab
runtime. After the workflow runs end to end, choose **"full"** if you want
cleaner samples.

In Colab, a GPU is helpful but not required:
**Runtime → Change runtime type → T4 GPU**. The code also supports CPU and
Apple Silicon; there are no hard-coded CUDA calls.

In [ ]:
import copy
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import FashionMNIST
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid
from tqdm.auto import tqdm

MODE = "quick"  # Use "full" only after the quick workflow succeeds.
SEED = 1

RUN_CONFIGS = {
    "quick": {
        "dataset_size": 12_000,
        "batch_size": 256,
        "vae_epochs": 2,
        "diffusion_epochs": 4,
        "diffusion_base_channels": 24,
        "sampling_budgets": [25, 50, 100],
        "validation_size": 2_000,
        "num_workers": 0,
    },
    "full": {
        "dataset_size": None,
        "batch_size": 256,
        "vae_epochs": 6,
        "diffusion_epochs": 8,
        "diffusion_base_channels": 32,
        "sampling_budgets": [25, 50, 100, 200],
        "validation_size": 5_000,
        "num_workers": 2,
    },
}
if MODE not in RUN_CONFIGS:
    raise ValueError(f"Unknown run mode: {MODE!r}")
cfg = RUN_CONFIGS[MODE]
QUICK_MODE = MODE == 'quick'


def choose_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = choose_device()


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


set_seed()
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

print(f"Mode: {MODE}")
print(f"Device: {DEVICE}")
print("Configuration:", cfg)

## 2. Data: one consistent 28 × 28 path

FashionMNIST contains 60,000 training images across ten clothing categories.
Both models use the native \(1\times28\times28\) shape. The VAE receives values
in \([0,1]\); diffusion converts the same batches to \([-1,1]\) immediately
before noising. This fixes the source notebook's mismatch between 28 × 28
training images and 32 × 32 generated images.

Both modes first carve a deterministic validation holdout from the 60,000
training images; quick mode then chooses its smaller training subset from the
remaining images. Reconstructions use the holdout, never a training batch.
The official FashionMNIST test split remains reserved for future final
evaluation. Labels are returned by the dataset but are not used by either
model.

In [ ]:
full_train = FashionMNIST(
    root="./data", train=True, download=True, transform=ToTensor()
)
subset_generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(full_train), generator=subset_generator)
validation_indices = permutation[: cfg["validation_size"]]
remaining_indices = permutation[cfg["validation_size"] :]
selected_indices = (
    remaining_indices
    if cfg["dataset_size"] is None
    else remaining_indices[: cfg["dataset_size"]]
)
train_data = Subset(full_train, selected_indices.tolist())
validation_data = Subset(full_train, validation_indices.tolist())
fixed_eval_data = Subset(full_train, validation_indices[:16].tolist())


def make_train_loader(seed=SEED):
    return DataLoader(
        train_data,
        batch_size=cfg["batch_size"],
        shuffle=True,
        drop_last=True,
        num_workers=cfg["num_workers"],
        pin_memory=(DEVICE.type == "cuda"),
        generator=torch.Generator().manual_seed(seed),
    )


fixed_eval_loader = DataLoader(fixed_eval_data, batch_size=16, shuffle=False)
fixed_images_01, _ = next(iter(fixed_eval_loader))
fixed_images_01 = fixed_images_01.to(DEVICE)


def show_grid(images, title, nrow=8, value_range=(0.0, 1.0), ax=None):
    grid = make_grid(
        images.detach().cpu(),
        nrow=nrow,
        padding=2,
        normalize=True,
        value_range=value_range,
    )
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 3))
    ax.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    ax.set_title(title)
    ax.axis("off")
    return ax


print(f"Training images: {len(train_data):,}")
print(f"Validation holdout: {len(validation_data):,} images")
print("Official FashionMNIST test split: reserved and not loaded")
print("Batch shape:", tuple(next(iter(make_train_loader()))[0].shape))
show_grid(fixed_images_01, "Fixed validation-holdout comparison batch")
plt.show()

## 3. Variational autoencoder

The encoder produces a mean and log variance. Reparameterization moves the
randomness into a standard-normal variable:

\[
z=\mu+\exp\left(\tfrac12\log\sigma^2\right)\epsilon,
\qquad \epsilon\sim N(0,I).
\]

We minimize the negative beta-ELBO per example,

\[
\mathcal L_\beta
=\operatorname{BCE}(x,\hat x)
+\beta D_{KL}\left(q(z\mid x)\Vert N(0,I)\right).
\]

Latent dimension and beta are the tuning knobs; the architecture, objective,
and training loop are prefilled.

In [ ]:
class VAE(nn.Module):
    def __init__(self, z_dims=4, input_size=784, num_hidden=128):
        super().__init__()
        self.z_dims = z_dims
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden, num_hidden),
            nn.ReLU(),
        )
        self.mu = nn.Linear(num_hidden, z_dims)
        self.logvar = nn.Linear(num_hidden, z_dims)
        self.decoder = nn.Sequential(
            nn.Linear(z_dims, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden, num_hidden),
            nn.ReLU(),
            nn.Linear(num_hidden, input_size),
            nn.Sigmoid(),
        )

    def encode(self, x):
        hidden = self.encoder(x)
        return self.mu(hidden), self.logvar(hidden)

    @staticmethod
    def reparameterize(mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)

    def forward(self, x):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar)), mu, logvar


def vae_objective(reconstructed, x, mu, logvar, beta_vae):
    reconstruction = F.binary_cross_entropy(
        reconstructed.flatten(1), x.flatten(1), reduction="none"
    ).sum(dim=1)
    kl = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1)
    return (
        (reconstruction + beta_vae * kl).mean(),
        reconstruction.mean(),
        kl.mean(),
    )


def train_vae(model, beta_vae, epochs, loader):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    history = {"loss": [], "reconstruction": [], "kl": []}
    for epoch in range(1, epochs + 1):
        model.train()
        totals = {key: 0.0 for key in history}
        count = 0
        for images, _ in loader:
            images = images.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            reconstructed, mu, logvar = model(images)
            loss, reconstruction, kl = vae_objective(
                reconstructed, images, mu, logvar, beta_vae
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            batch_size = images.shape[0]
            for key, value in (
                ("loss", loss),
                ("reconstruction", reconstruction),
                ("kl", kl),
            ):
                totals[key] += value.item() * batch_size
            count += batch_size
        for key in history:
            history[key].append(totals[key] / count)
        print(
            f"VAE epoch {epoch:02d}/{epochs}: "
            f"loss={history['loss'][-1]:.3f}, "
            f"reconstruction={history['reconstruction'][-1]:.3f}, "
            f"KL={history['kl'][-1]:.3f}"
        )
    return history

### Fixed submission checkpoint: KL diagnostic

For one two-dimensional posterior use
\(\mu=(0,1)\) and \(\log\sigma^2=(0,\log4)\). The cell computes the sum over
the two coordinates for one example. Enter the printed value in edX. It is
independent of training and deterministic on every device.

In [ ]:
diagnostic_mu = torch.tensor([[0.0, 1.0]], dtype=torch.float64)
diagnostic_logvar = torch.log(
    torch.tensor([[1.0, 4.0]], dtype=torch.float64)
)
diagnostic_kl = -0.5 * torch.sum(
    1 + diagnostic_logvar
    - diagnostic_mu.pow(2)
    - diagnostic_logvar.exp()
)
expected_kl = 0.5 * (4.0 - math.log(4.0))
assert math.isclose(diagnostic_kl.item(), expected_kl, abs_tol=1e-12)
EXPECTED_KL_ROUNDED = 1.30685
assert round(diagnostic_kl.item(), 5) == EXPECTED_KL_ROUNDED
print(f"M5-C1 total KL: {diagnostic_kl.item():.5f}")

### Controlled VAE sweep

The supplied sweep compares three beta values at fixed latent dimension and
three latent dimensions at fixed beta. The baseline is shared, so only five
models are trained. Every run receives the same data order and seed.

Inspect posterior-mean reconstructions and fixed-seed prior samples. Smaller
beta often improves reconstruction while permitting a less organized latent
distribution; larger beta strengthens prior matching but can discard useful
detail. These are trends to investigate, not graded measured outcomes.

In [ ]:
VAE_SETTINGS = [
    {"name": "beta=0.1", "z_dims": 4, "beta_vae": 0.1},
    {"name": "baseline", "z_dims": 4, "beta_vae": 1.0},
    {"name": "beta=2.0", "z_dims": 4, "beta_vae": 2.0},
    {"name": "z=2", "z_dims": 2, "beta_vae": 1.0},
    {"name": "z=8", "z_dims": 8, "beta_vae": 1.0},
]

vae_runs = {}
for setting in VAE_SETTINGS:
    print(
        f"\nTraining {setting['name']} "
        f"(z={setting['z_dims']}, beta={setting['beta_vae']})"
    )
    set_seed(SEED)
    model = VAE(z_dims=setting["z_dims"]).to(DEVICE)
    history = train_vae(
        model,
        setting["beta_vae"],
        cfg["vae_epochs"],
        make_train_loader(SEED),
    )
    vae_runs[setting["name"]] = {
        **setting, "model": model.eval(), "history": history
    }


def fixed_normal(shape, seed):
    generator = torch.Generator(device="cpu").manual_seed(seed)
    return torch.randn(shape, generator=generator).to(DEVICE)


fig, axes = plt.subplots(len(VAE_SETTINGS), 2, figsize=(11, 12))
with torch.inference_mode():
    for row, setting in enumerate(VAE_SETTINGS):
        model = vae_runs[setting["name"]]["model"]
        mu, _ = model.encode(fixed_images_01)
        reconstructions = model.decode(mu)
        samples = model.decode(fixed_normal((16, model.z_dims), 2025))
        show_grid(
            reconstructions,
            f"{setting['name']}: reconstructions",
            ax=axes[row, 0],
        )
        show_grid(
            samples,
            f"{setting['name']}: prior samples",
            ax=axes[row, 1],
        )
plt.tight_layout()
plt.show()

In [ ]:
def plot_latent_grid(model, dim_i=0, dim_j=1, points=10, limit=2.5):
    if dim_i == dim_j or max(dim_i, dim_j) >= model.z_dims:
        raise ValueError("Choose two distinct valid latent dimensions.")
    values = torch.linspace(-limit, limit, points, device=DEVICE)
    z = torch.zeros(points * points, model.z_dims, device=DEVICE)
    index = 0
    for vertical in reversed(values):
        for horizontal in values:
            z[index, dim_i] = horizontal
            z[index, dim_j] = vertical
            index += 1
    with torch.inference_mode():
        decoded = model.decode(z)
    grid = make_grid(decoded.cpu(), nrow=points, padding=1)
    plt.figure(figsize=(7, 7))
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray", vmin=0, vmax=1)
    plt.title("Baseline VAE latent grid")
    plt.axis("off")
    plt.show()


plot_latent_grid(vae_runs["baseline"]["model"])

## 4. Diffusion forward process

For \(T=200\), define

\[
\beta_t\in[10^{-4},0.02],\quad
\alpha_t=1-\beta_t,\quad
\bar\alpha_t=\prod_{s=0}^{t}\alpha_s.
\]

A clean \(x_0\in[-1,1]\) can be noised directly:

\[
x_t=\sqrt{\bar\alpha_t}x_0+
\sqrt{1-\bar\alpha_t}\epsilon,\qquad\epsilon\sim N(0,I).
\]

The compact convolutional model predicts the injected noise from \((x_t,t)\).
It replaces the source notebook's externally versioned U-Net dependency.

In [ ]:
TOTAL_TIMESTEPS = 200


def make_schedule(total_timesteps, device):
    betas = torch.linspace(
        1e-4, 0.02, total_timesteps, device=device, dtype=torch.float32
    )
    alphas = 1.0 - betas
    alpha_bars = torch.cumprod(alphas, dim=0)
    return {
        "betas": betas,
        "alphas": alphas,
        "alpha_bars": alpha_bars,
        "sqrt_alpha_bars": torch.sqrt(alpha_bars),
        "sqrt_one_minus_alpha_bars": torch.sqrt(1.0 - alpha_bars),
    }


schedule = make_schedule(TOTAL_TIMESTEPS, DEVICE)


def extract(values, timesteps, shape):
    return values.gather(0, timesteps).reshape(
        timesteps.shape[0], *([1] * (len(shape) - 1))
    )


def q_sample(x_0, timesteps, epsilon):
    return (
        extract(schedule["sqrt_alpha_bars"], timesteps, x_0.shape) * x_0
        + extract(
            schedule["sqrt_one_minus_alpha_bars"], timesteps, x_0.shape
        ) * epsilon
    )


fixed_clean = fixed_images_01[:8] * 2.0 - 1.0
fixed_epsilon = fixed_normal(fixed_clean.shape, 303)
levels = [0, 50, 100, 199]
fig, axes = plt.subplots(1, len(levels), figsize=(13, 3))
for ax, level in zip(axes, levels):
    timesteps = torch.full(
        (fixed_clean.shape[0],), level, device=DEVICE, dtype=torch.long
    )
    show_grid(
        q_sample(fixed_clean, timesteps, fixed_epsilon),
        f"Forward process: t={level}",
        value_range=(-1, 1),
        ax=ax,
    )
plt.tight_layout()
plt.show()

## 5. Compact time-conditioned denoiser

The model uses a sinusoidal time embedding, a 28→14→28 convolutional path,
and a skip connection. Its output has exactly the same shape as its input.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dimension):
        super().__init__()
        if dimension % 2:
            raise ValueError("Time dimension must be even.")
        self.dimension = dimension

    def forward(self, timesteps):
        half = self.dimension // 2
        frequencies = torch.exp(
            -math.log(10_000)
            * torch.arange(half, device=timesteps.device, dtype=torch.float32)
            / max(half - 1, 1)
        )
        angles = timesteps.float()[:, None] * frequencies[None, :]
        return torch.cat((angles.sin(), angles.cos()), dim=1)


class TimeBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dimension):
        super().__init__()
        groups = 8 if out_channels % 8 == 0 else 4
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(groups, out_channels)
        self.time_projection = nn.Linear(time_dimension, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_channels)
        self.skip = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Conv2d(in_channels, out_channels, 1)
        )

    def forward(self, x, time_embedding):
        hidden = self.norm1(self.conv1(x))
        hidden = hidden + self.time_projection(time_embedding)[:, :, None, None]
        hidden = F.silu(hidden)
        hidden = self.norm2(self.conv2(hidden))
        return F.silu(hidden + self.skip(x))


class SmallDenoiser(nn.Module):
    def __init__(self, base_channels=24, time_dimension=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(time_dimension),
            nn.Linear(time_dimension, time_dimension),
            nn.SiLU(),
            nn.Linear(time_dimension, time_dimension),
        )
        self.input_block = TimeBlock(1, base_channels, time_dimension)
        self.downsample = nn.Conv2d(
            base_channels, 2 * base_channels, 4, stride=2, padding=1
        )
        self.down_block = TimeBlock(
            2 * base_channels, 2 * base_channels, time_dimension
        )
        self.middle_block = TimeBlock(
            2 * base_channels, 2 * base_channels, time_dimension
        )
        self.upsample = nn.ConvTranspose2d(
            2 * base_channels, base_channels, 4, stride=2, padding=1
        )
        self.output_block = TimeBlock(
            2 * base_channels, base_channels, time_dimension
        )
        self.output = nn.Conv2d(base_channels, 1, 3, padding=1)

    def forward(self, x, timesteps):
        time_embedding = self.time_mlp(timesteps)
        skip = self.input_block(x, time_embedding)
        hidden = self.downsample(skip)
        hidden = self.down_block(hidden, time_embedding)
        hidden = self.middle_block(hidden, time_embedding)
        hidden = self.upsample(hidden)
        if hidden.shape[-2:] != skip.shape[-2:]:
            hidden = F.interpolate(
                hidden, size=skip.shape[-2:], mode="bilinear",
                align_corners=False
            )
        hidden = torch.cat((hidden, skip), dim=1)
        hidden = self.output_block(hidden, time_embedding)
        return self.output(hidden)


set_seed(SEED)
denoiser = SmallDenoiser(
    base_channels=cfg["diffusion_base_channels"]
).to(DEVICE)
with torch.inference_mode():
    test_x = torch.zeros(2, 1, 28, 28, device=DEVICE)
    test_t = torch.tensor([0, TOTAL_TIMESTEPS - 1], device=DEVICE)
    assert denoiser(test_x, test_t).shape == test_x.shape
print(f"Denoiser parameters: {sum(p.numel() for p in denoiser.parameters()):,}")

In [ ]:
@torch.no_grad()
def update_ema(ema_model, online_model, decay=0.99):
    online_parameters = dict(online_model.named_parameters())
    for name, ema_parameter in ema_model.named_parameters():
        ema_parameter.mul_(decay).add_(
            online_parameters[name], alpha=1.0 - decay
        )
    online_buffers = dict(online_model.named_buffers())
    for name, ema_buffer in ema_model.named_buffers():
        ema_buffer.copy_(online_buffers[name])


def train_diffusion(model, epochs, loader):
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
    ema_model = copy.deepcopy(model).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = 0.0
        count = 0
        progress = tqdm(loader, desc=f"Diffusion epoch {epoch}/{epochs}")
        for images, _ in progress:
            x_0 = images.to(DEVICE, non_blocking=True) * 2.0 - 1.0
            batch_size = x_0.shape[0]
            timesteps = torch.randint(
                0, TOTAL_TIMESTEPS, (batch_size,), device=DEVICE
            )
            epsilon = torch.randn_like(x_0)
            x_t = q_sample(x_0, timesteps, epsilon)
            optimizer.zero_grad(set_to_none=True)
            predicted_epsilon = model(x_t, timesteps)
            loss = F.mse_loss(predicted_epsilon, epsilon)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            update_ema(ema_model, model)
            loss_sum += loss.item() * batch_size
            count += batch_size
            progress.set_postfix(mse=f"{loss.item():.4f}")
        history.append(loss_sum / count)
        print(
            f"Diffusion epoch {epoch:02d}/{epochs}: "
            f"noise MSE={history[-1]:.4f}"
        )
    return ema_model.eval(), history


set_seed(SEED)
ema_denoiser, diffusion_history = train_diffusion(
    denoiser,
    cfg["diffusion_epochs"],
    make_train_loader(SEED + 10),
)
plt.figure(figsize=(5, 3))
plt.plot(range(1, len(diffusion_history) + 1), diffusion_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training noise-prediction MSE")
plt.title("Diffusion training history (ungraded)")
plt.grid(alpha=0.25)
plt.show()

## 6. Valid reduced-step sampling with DDIM

An ancestral DDPM update is defined between adjacent times, so merely skipping
indices in that update would be invalid. We instead use deterministic
\((\eta=0)\) DDIM. At selected time \(t\),

\[
\hat x_0 =
\frac{x_t-\sqrt{1-\bar\alpha_t}\epsilon_\theta(x_t,t)}
{\sqrt{\bar\alpha_t}},
\]

and at the next selected time \(s<t\),

\[
x_s=\sqrt{\bar\alpha_s}\hat x_0+
\sqrt{1-\bar\alpha_s}\epsilon_\theta(x_t,t).
\]

This remains defined for a respaced sequence. Holding the checkpoint and
starting noise fixed makes the comparison deterministic. More steps mean more
denoiser calls; quality may improve or plateau, but an individual image is not
guaranteed to improve.

In [ ]:
def ddim_timesteps(total_timesteps, sampling_steps):
    if not 1 <= sampling_steps <= total_timesteps:
        raise ValueError("Invalid sampling-step budget.")
    values = torch.linspace(
        total_timesteps - 1, 0, sampling_steps
    ).round().long()
    if torch.unique_consecutive(values).numel() != sampling_steps:
        raise RuntimeError("Respaced timestep sequence has duplicates.")
    return values.tolist()


@torch.inference_mode()
def ddim_sample(model, initial_noise, sampling_steps, keep_history=False):
    model.eval()
    x = initial_noise.clone().to(DEVICE)
    times = ddim_timesteps(TOTAL_TIMESTEPS, sampling_steps)
    next_times = times[1:] + [-1]
    history = [x.detach().cpu()] if keep_history else None
    snapshots = set(
        np.linspace(0, len(times) - 1, min(5, len(times)), dtype=int)
    )

    for position, (time_value, next_value) in enumerate(
        zip(times, next_times)
    ):
        timesteps = torch.full(
            (x.shape[0],), time_value, device=DEVICE, dtype=torch.long
        )
        predicted_epsilon = model(x, timesteps)
        alpha_bar_t = schedule["alpha_bars"][time_value]
        predicted_x_0 = (
            x - torch.sqrt(1.0 - alpha_bar_t) * predicted_epsilon
        ) / torch.sqrt(alpha_bar_t)
        predicted_x_0 = predicted_x_0.clamp(-1.0, 1.0)
        if next_value < 0:
            x = predicted_x_0
        else:
            alpha_bar_next = schedule["alpha_bars"][next_value]
            x = (
                torch.sqrt(alpha_bar_next) * predicted_x_0
                + torch.sqrt(1.0 - alpha_bar_next) * predicted_epsilon
            )
        if keep_history and position in snapshots:
            history.append(x.detach().cpu())
    return x.clamp(-1.0, 1.0), history


for budget in cfg["sampling_budgets"]:
    times = ddim_timesteps(TOTAL_TIMESTEPS, budget)
    assert len(times) == budget
    assert times[0] == TOTAL_TIMESTEPS - 1 and times[-1] == 0
print("Validated DDIM budgets:", cfg["sampling_budgets"])

### Controlled step-budget comparison

Every run below starts from the same 16 noise images and uses the same trained
EMA checkpoint. The number of DDIM updates is the intended variable. Runtime
is measured for context but is not graded.

In [ ]:
def synchronize_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps" and hasattr(torch, "mps"):
        torch.mps.synchronize()


fixed_diffusion_noise = fixed_normal((16, 1, 28, 28), 2025)
sampling_results = {}
for budget in cfg["sampling_budgets"]:
    synchronize_device()
    start = time.perf_counter()
    generated, history = ddim_sample(
        ema_denoiser,
        fixed_diffusion_noise,
        budget,
        keep_history=(budget == max(cfg["sampling_budgets"])),
    )
    synchronize_device()
    elapsed = time.perf_counter() - start
    sampling_results[budget] = {
        "images": generated,
        "history": history,
        "elapsed_seconds": elapsed,
    }
    print(
        f"{budget:3d} DDIM steps = {budget:3d} denoiser calls per batch; "
        f"measured runtime={elapsed:.2f} s (ungraded)"
    )

fig, axes = plt.subplots(
    len(cfg["sampling_budgets"]), 1,
    figsize=(11, 2.7 * len(cfg["sampling_budgets"]))
)
axes = np.atleast_1d(axes)
for ax, budget in zip(axes, cfg["sampling_budgets"]):
    images_01 = (sampling_results[budget]["images"] + 1.0) / 2.0
    show_grid(images_01, f"DDIM: {budget} denoiser evaluations", ax=ax)
plt.tight_layout()
plt.show()

largest_budget = max(cfg["sampling_budgets"])
history = sampling_results[largest_budget]["history"]
fig, axes = plt.subplots(1, len(history), figsize=(3 * len(history), 3))
for index, (ax, state) in enumerate(zip(axes, history)):
    image = ((state[0, 0] + 1.0) / 2.0).clamp(0, 1)
    ax.imshow(image.numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title("Initial noise" if index == 0 else f"Snapshot {index}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Fixed-seed model-family comparison

Compare fixed-seed samples from the baseline VAE and the largest diffusion
budget. Inspect sharpness, diversity, repeated or implausible outputs, and
sampling cost. Do not reduce the evidence to “one family always wins,” and
check whether your conclusion changes in full mode.

In [ ]:
baseline_vae = vae_runs["baseline"]["model"]
with torch.inference_mode():
    z = fixed_normal((16, baseline_vae.z_dims), 2025)
    vae_samples = baseline_vae.decode(z)
diffusion_samples = (sampling_results[largest_budget]["images"] + 1.0) / 2.0

fig, axes = plt.subplots(2, 1, figsize=(11, 5.5))
show_grid(
    vae_samples,
    "Baseline VAE: one decoder evaluation per sample",
    ax=axes[0],
)
show_grid(
    diffusion_samples,
    f"Diffusion: {largest_budget} denoiser evaluations per sample",
    ax=axes[1],
)
plt.tight_layout()
plt.show()

## 8. What to record and reflect

Only **M5-C1** is a numeric edX submission. The other values below describe
this particular run and are explicitly ungraded.

Use your figures to answer the conceptual checks:

1. If smaller beta improves reconstruction but makes prior samples less
   coherent, what changed in the reconstruction–regularization tradeoff?
2. When comparing 25 and 100 valid DDIM steps with checkpoint and initial
   noise held fixed, what changes deterministically, and what remains an
   empirical quality question?

For a possible Final Project direction, choose one observed failure mode and
write down a controlled intervention, a fixed comparison, and an evaluation
measure that would test it.

In [ ]:
print("=" * 68)
print("MODULE 5 LAB REPORT")
print("=" * 68)
print(f"M5-C1 total KL (submit this number): {diagnostic_kl.item():.5f}")
print()
print(f"Run mode: {MODE}")
print("VAE final values below are measured and UNGRADED:")
for setting in VAE_SETTINGS:
    run = vae_runs[setting["name"]]
    hist = run["history"]
    print(
        f"  {setting['name']:>8s}: "
        f"reconstruction={hist['reconstruction'][-1]:.3f}, "
        f"KL={hist['kl'][-1]:.3f}"
    )
print()
print("Diffusion values below are measured and UNGRADED:")
print(f"  final training noise MSE={diffusion_history[-1]:.4f}")
for budget in cfg["sampling_budgets"]:
    print(
        f"  {budget:3d} steps = {budget:3d} denoiser calls per batch; "
        f"measured runtime={sampling_results[budget]['elapsed_seconds']:.2f} s"
    )
print()
print("Measured losses, runtimes, and visual judgments are not graded.")
print("Return to edX for the beta-sweep and DDIM-design questions.")